# Testing LLM APIs
Welcome to this practical session. In this notebook, we will explore how to interact with the production OpenAI API, manipulate hyperparameters like temperature, and build automated testing assertions for validating structured responses.

In [1]:
import os
import os.path as osp

from pathlib import Path
from pprint import pprint

import sys

root = Path.cwd().parent 
if str(root) not in sys.path:    
    sys.path.append(str(root))

## Installation & Environment Setup
First, we need to install the official OpenAI SDK and configure our secure API token environment variable.

In [2]:

import os
import json
from openai import OpenAI
import os 

# Instruct students to input their real OpenAI API key
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "")

# Initialize the standard production client
client = OpenAI()
print("OpenAI Production Client Initialized!")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

## Managing Hyperparameters: Temperature Determinism
Let's see how setting a low temperature (highly deterministic) vs. a high temperature (highly creative) impacts raw API output variability.

In [2]:
prompt = "Generate a random three-word fantasy book title."

In [3]:
print("--- Testing Deterministic Output (temperature=0.0) ---")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0
    )
    print(f"Run {i+1}: {response.choices.message.content.strip()}")



--- Testing Deterministic Output (temperature=0.0) ---


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
print("\n--- Testing Creative Output (temperature=1.2) ---")
for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.2
    )
    print(f"Run {i+1}: {response.choices.message.content.strip()}")

## 3. Automated Assertions and Guardrails
When integrating LLMs into applications, you must test that outputs meet structural requirements. Let's write an assertion test ensuring the model successfully adheres to JSON mode.

In [ ]:
def test_json_output_mode():
    prompt = "Return a JSON object classifying Python as an advanced language. Use keys: 'language' and 'level'."
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        response_format={ "type": "json_object" },
        messages=[{"role": "user", "content": prompt}]
    )
    
    raw_text = response.choices.message.content
    
    try:
        parsed = json.loads(raw_text)
        assert "language" in parsed, "Missing required key: 'language'"
        assert "level" in parsed, "Missing required key: 'level'"
        print("✅ Pass: Response is valid JSON and contains all required keys.")
        print("Output:", parsed)
    except json.JSONDecodeError:
        print("❌ Fail: Response is not valid JSON.")
    except AssertionError as e:
        print(f"❌ Fail: {str(e)}")

test_json_output_mode()

## 4. Student Exercise
Write an automated guardrail function called `test_content_guardrail(prompt, banned_word)`. 
It must query the `gpt-4o-mini` model, check if the `banned_word` exists anywhere inside the generated response text, and raise an error or print a failure if it breaches safety constraints.

In [ ]:
# TODO: Complete the student exercise function
def test_content_guardrail(prompt, banned_word):
    # 1. Generate response using client.chat.completions.create
    # 2. Check if banned_word is in the response text
    # 3. Print pass or fail
    pass

# Test your function here:
# test_content_guardrail("Tell me a story about gold coins", "gold")

### Solution Hint
Uncomment and run below to check your answer if you get stuck.

In [ ]:
# def test_content_guardrail(prompt, banned_word):
#     response = client.chat.completions.create(
#         model="gpt-4o-mini",
#         messages=[{"role": "user", "content": prompt}]
#     )
#     text = response.choices.message.content.lower()
#     if banned_word.lower() in text:
#         print(f"❌ Fail: Guardrail breached! Found banned word '{banned_word}'")
#     else:
#         print("✅ Pass: Output cleared safety guardrail.")
#
# test_content_guardrail("Tell me a story about gold coins", "gold")